In [1]:
import torch
from torch_geometric.nn import GATv2Conv
from torch_geometric.data import Data
import torch.autograd.profiler as profiler
from gflownet.utils import log_memory_usage
from scipy.io import mmread  # assuming your matrices are in Matrix Market format
import pytorch_lightning as pl
import numpy as np
import os
from gflownet.validate import load_mtx_file, load_vector_mtx
from scipy.sparse import csr_matrix, csc_matrix
import gc


ilu_matrix = mmread('/Users/tonylizza/Documents/Machine_Learning/Thesis_Coding/gflownet-spai/data/medium_ILU/ilu_matrices/pores_3_ilu.mtx').tocoo()
ilu_values = torch.tensor(ilu_matrix.data, dtype=torch.float32)
ilu_indices = torch.tensor([ilu_matrix.row, ilu_matrix.col], dtype=torch.long)
ilu_sparse_tensor = torch.sparse_coo_tensor(ilu_indices, ilu_values, ilu_matrix.shape)

# Create PyTorch Geometric `Data` object for ILU preconditioner
edge_index = torch.stack([torch.tensor(ilu_matrix.row, dtype=torch.long), torch.tensor(ilu_matrix.col, dtype=torch.long)], dim=0)
edge_attr = torch.tensor(ilu_matrix.data, dtype=torch.float32)
x = torch.ones((ilu_matrix.shape[0], 1))  # Example node features

#ilu_data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
'''
# Dummy data with 4 nodes and 3 features
x = torch.randn((4, 3))
edge_index = torch.tensor([[0, 1, 2, 3], [1, 2, 0, 0]], dtype=torch.long)
edge_attr = torch.randn((4, 1))
'''

# Define the GATv2Conv layer
gat_layer = GATv2Conv(in_channels=-1, out_channels=4, heads=2, edge_dim=1)

# Profile memory usage for isolated GATv2Conv layer

for i in range(1000):
    with profiler.profile(profile_memory=True, record_shapes=True) as prof:
        output = gat_layer(x, edge_index, edge_attr)
        log_memory_usage(f"For {i}: ")
        
    #print(prof.key_averages().table(sort_by="self_cpu_memory_usage", row_limit=10))


/var/folders/df/jtzlym5n0d9b8ttc8phbv7300000gn/T/ipykernel_40246/431545050.py:17: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1719361045918/work/torch/csrc/utils/tensor_new.cpp:277.)
  ilu_indices = torch.tensor([ilu_matrix.row, ilu_matrix.col], dtype=torch.long)


[For 0: ] CPU Memory Usage: 311.62 MB; VMS: 34981.60 MB
[For 1: ] CPU Memory Usage: 311.29 MB; VMS: 34980.29 MB
[For 2: ] CPU Memory Usage: 311.08 MB; VMS: 34979.62 MB
[For 3: ] CPU Memory Usage: 311.33 MB; VMS: 34979.62 MB
[For 4: ] CPU Memory Usage: 311.87 MB; VMS: 34980.62 MB
[For 5: ] CPU Memory Usage: 312.34 MB; VMS: 34981.62 MB
[For 6: ] CPU Memory Usage: 312.68 MB; VMS: 34981.62 MB
[For 7: ] CPU Memory Usage: 312.93 MB; VMS: 34981.62 MB
[For 8: ] CPU Memory Usage: 313.26 MB; VMS: 34981.62 MB
[For 9: ] CPU Memory Usage: 313.67 MB; VMS: 34981.62 MB
[For 10: ] CPU Memory Usage: 314.01 MB; VMS: 34981.62 MB
[For 11: ] CPU Memory Usage: 314.01 MB; VMS: 34981.62 MB
[For 12: ] CPU Memory Usage: 314.38 MB; VMS: 34981.62 MB
[For 13: ] CPU Memory Usage: 314.58 MB; VMS: 34981.62 MB
[For 14: ] CPU Memory Usage: 314.80 MB; VMS: 34981.62 MB
[For 15: ] CPU Memory Usage: 314.94 MB; VMS: 34981.62 MB
[For 16: ] CPU Memory Usage: 315.12 MB; VMS: 34981.62 MB
[For 17: ] CPU Memory Usage: 315.20 MB; V

KeyboardInterrupt: 